In [172]:
import os
import requests
import numpy as np
import pandas as pd
from selenium import webdriver
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI

In [173]:
# Load environment variables
load_dotenv()

# Initialize OpenAI client with API key
client = OpenAI()
OpenAI.api_key = os.getenv('OPENAI_API_KEY')

In [174]:
# Load the dataset from a CSV file
def load_data(file_path):
    """Loads a CSV file into a pandas DataFrame.
    
    Args:
        file_path (str): The path to the CSV file.
    
    Returns:
        DataFrame: A pandas DataFrame containing the loaded data.
    """
    return pd.read_csv(file_path)

In [175]:
# Function to filter out rows containing social media URLs
def filter_social_media_urls(data_frame):
    """Filters out rows from the DataFrame where the 'source_url' contains social media domains.
    
    Args:
        data_frame (DataFrame): The DataFrame to filter.
    
    Returns:
        DataFrame: A filtered DataFrame without social media URLs.
    """
    # List of social media domains to filter out
    social_media_domains = [
        'facebook.com',
        'twitter.com',
        'x.com',
        'linkedin.com',
        'instagram.com',
        'tiktok.com',
        'youtube.com',
        'telegram.com'
    ]

    # Compile the pattern for filtering URLs
    pattern = '|'.join(social_media_domains)
    
    # Convert 'source_url' column to string and handle NaN values
    df['urls'] = df['source_url'].astype(str)
    
    # Filter out rows containing social media URLs
    filtered_df = df[~df['source_url'].str.contains(pattern, na=False)]
    
    # Drop rows with NaN in 'source_url' and reset the index
    filtered_df = filtered_df.dropna(subset=['source_url']).reset_index(drop=True)
    
    return filtered_df

In [176]:
def scrape_website(url):
    """Scrapes the HTML content of a website given its URL.
    
    Args:
        url (str): The URL of the website to scrape.
    
    Returns:
        BeautifulSoup: A BeautifulSoup object containing the parsed HTML.
    """
    # Initialize the web driver (Chrome in this case)
    driver = webdriver.Chrome()  # Ensure you have the ChromeDriver installed

    try:
        driver.get(url)  # Open the specified URL
        html_content = driver.page_source  # Get the HTML content of the page
        soup = BeautifulSoup(html_content, 'html.parser')  # Parse the HTML with BeautifulSoup
    finally:
        driver.quit()  # Ensure the driver is closed after the operation
    
    return soup  # Return the parsed HTML

In [185]:
def generate_prompt(page_text, links):
    """Generates a prompt for extracting monetization-related information from a webpage.
    
    Args:
        page_text (str): The text content of the webpage to analyze.
        links (list): A list of URLs found on the webpage.
    
    Returns:
        str: A formatted prompt for data extraction in JSON format.
    """
    
    prompt = f"""
You are tasked with identifying monetization-related information from websites. The website content may be in English or another language, but your goal is to extract and translate all relevant details into English. Analyze the provided **Webpage text:** 
{page_text} and **links:** {links}, and output the extracted data in JSON format.

   - **Donation Links:** URLs pointing to donation pages or forms.
   - **Donation Methods:** Accepted payment methods for donations (e.g., PayPal, credit cards, cryptocurrency, direct donation).
   - **Suggested Donation Amounts:** Preset or recommended amounts for donations.
   - **Recurring Donations:** Whether recurring donations are supported.
   - **Donation-Related Text:** Extract and translate any promotional text encouraging donations (e.g., 'Support us', 'Donate now').
   - **Donation Buttons/Widgets:** Describe any buttons or widgets that facilitate donations.
   - **Donation Section Location:** Where donation-related elements are located on the page (e.g., header, footer, sidebar).
   - **Third-Party Platforms:** Mention any external platforms used for donations (e.g., Patreon, GoFundMe).
   - **Call-to-Action:** Highlight any prominent donation calls-to-action and translate them.
   - **Payment Platforms:** Identify any integrated payment platforms (e.g., Stripe, PayPal).
   - **Payment Button/Widget Location:** Location of payment-related buttons or widgets on the page (e.g., top, bottom, sidebar).
   - **Accepted Payment Methods:** List the payment methods accepted for purchases or subscriptions.
   - **Pricing Models:** Specify any pricing models used (e.g., subscription plans, pay-per-article).
   - **Currency Support:** Types of currencies accepted for payments or donations.
   - **Subscription Plans:** Details about available subscription options, pricing tiers, and benefits (translated into English).

If no relevant information is found, return a JSON structure with all relevant fields. The output should only be the JSON data, with no extra text.
"""
    return prompt

In [178]:
def generate_completions(prompt, model='gpt-4', max_tokens=2000):
    """Sends a request to the OpenAI API to generate completions based on the given prompt.

    Args:
        prompt (str): The text prompt to be used by the model.
        model (str): The model name to be used. Defaults to 'gpt-4'.
        max_tokens (int): The maximum number of tokens to generate. Defaults to 2000.

    Returns:
        str: The generated completion from the model.
    """
    # Send a request to the OpenAI API for completion
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=max_tokens  # Specify the maximum number of tokens
    )

    # Return the parsed message containing the generated completion
    return completion.choices[0].message.content

In [179]:
empty_template = {
    "MonetizationData": {
        "DonationLinks": [],
        "DonationMethods": [],
        "SuggestedDonationAmounts": [],
        "RecurringDonationsOption": "",
        "TextRelatedToDonations": [],
        "ButtonsOrWidgets": [],
        "PlacementOfDonationSection": "",
        "ThirdPartyPlatforms": [],
        "CallToActionMessages": [],
        "AcceptedPaymentMethods": [],
        "PricingModels": [],
        "CurrencySupport": [],
        "SubscriptionPlans": []
    }
}

In [180]:
file_path = "../data/ee24_simple-with-dates.csv"  # Path to the CSV file
df = load_data(file_path)  # Load the data into a DataFrame
df.head()

,Unnamed: 0,type,url,headline,euRelation,countryOfOrigin,claimreviewed,source_url,date_published_parsed
0,851,Debunk,https://verifica.efe.com/persona-ondea-bandera...,The person waving a Palestinian flag on a balc...,Direct,ES,King Frederick X of Denmark waves a Palestinia...,https://www.facebook.com/groups/82495245437533...,2024-05-14 16:01:56.000
1,846,Debunk,https://demagog.org.pl/fake_news/pozar-w-warsz...,The fire in Warsaw was caused to lock us in ou...,Indirect,PL,The warnings sent on the occasion of the fires...,https://www.facebook.com/reel/1005762777834591,2024-05-14 13:30:30.000
2,852,Debunk,https://faktencheck.afp.com/doc.afp.com.34R34VN,The travel warning for Germany was not set to ...,Indirect,FR,USA sets travel warning for Germany to the hig...,https://www.facebook.com/melanie.marz.7/posts/...,2024-05-14 11:57:00.000
3,843,Debunk,https://faktograf.hr/2024/05/14/francuski-pilo...,"French pilots made a French, not a Russian fla...",Direct,HR,French pilots made a Russian flag in the sky o...,https://www.facebook.com/nbabaicbutkovic/video...,2024-05-14 10:11:31.000
4,842,Debunk,https://tenykerdes.afp.com/doc.afp.com.34RH6AD,The board of this 268 million HUF EU project a...,Indirect,HU,This EU information board is about a single to...,https://www.facebook.com/photo/?fbid=791220124...,2024-05-14 07:50:00.000


In [182]:
filtered_df = filter_social_media_urls(df)  # Filter the DataFrame
filtered_df.head()

,Unnamed: 0,type,url,headline,euRelation,countryOfOrigin,claimreviewed,source_url,date_published_parsed,urls
0,840,Factcheck,https://www.lessurligneurs.eu/programme-rn-pou...,RN program for European women: reaffirming the...,Direct,FR,reaffirm the superiority of the French Constit...,https://vivementle9juin.fr/projet/europe-qui-r...,2024-05-13 18:00:33.000,https://vivementle9juin.fr/projet/europe-qui-r...
1,833,Debunk,https://www.ellinikahoaxes.gr/2024/05/13/rosik...,Russian propaganda: Doctored video with false ...,Indirect,GR,The White House responsibly declared that Worl...,https://pravda-gr.com/world/2024/05/09/6152.html,2024-05-13 15:48:28.000,https://pravda-gr.com/world/2024/05/09/6152.html
2,821,Debunk,https://www.ostro.si/si/razkrinkavanje/objave/...,CO2 has caused climate change in the past,Indirect,SI,“It also shows that there is no evidence that ...,https://triglavmedia.si/novice/okolje-ekologij...,2024-05-13 04:54:54.000,https://triglavmedia.si/novice/okolje-ekologij...
3,817,Debunk,https://factcheck.bg/publikacii-podvezhdat-che...,Publications mislead that Ukraine has withdraw...,Indirect,BG,Ukraine has revoked or suspended compliance wi...,https://euronewsbulgaria.com/news/27068/kiev-c...,2024-05-10 06:00:18.000,https://euronewsbulgaria.com/news/27068/kiev-c...
4,793,Factcheck,https://www.ostro.si/si/razkrinkavanje/objave/...,Pensioners were also eligible for the energy s...,Indirect,SI,"“But you did not think about pensioners, but f...",https://www.dz-rs.si/wps/portal/Home/seje/evid...,2024-05-10 04:00:00.000,https://www.dz-rs.si/wps/portal/Home/seje/evid...


In [183]:
filtered_df['source_url']

0      https://vivementle9juin.fr/projet/europe-qui-r...
1       https://pravda-gr.com/world/2024/05/09/6152.html
2      https://triglavmedia.si/novice/okolje-ekologij...
3      https://euronewsbulgaria.com/news/27068/kiev-c...
4      https://www.dz-rs.si/wps/portal/Home/seje/evid...
                             ...                        
188    https://slobodnadalmacija.hr/vijesti/svijet/iz...
189    https://www.7dnevno.hr/vijesti/huti-projektilo...
190    https://epoha.com.hr/2023/12/31/ocajna-ukrajin...
191    https://mondo.rs/Info/Ekonomija/a1839222/Kucni...
192    https://www.lentata.com/eksperti-po-kibersigur...
Name: source_url, Length: 193, dtype: object

In [186]:
# Initialize a counter and a list to store results
count = 0
dfs = []

# Loop through each URL in the filtered DataFrame
for url in filtered_df['source_url']:
    # Scrape the website data
    website_data = scrape_website(url)
    
    # Extract text from the webpage
    page_text = website_data.get_text()  # Extracts all text from the webpage
    # Extract all links from the webpage
    links = [a['href'] for a in website_data.find_all('a', href=True)]
    
    # Generate a prompt for the OpenAI API
    prompt = generate_prompt(page_text, links)
    
    # Get the completion data from the model
    response_data = generate_completions(prompt, model='gpt-4', max_tokens=2000)
    response_data = response_data.replace("```json\n", "").replace("```", "")
    
    # Print the raw data response for debugging
    print(response_data)
    
    # If no data is returned, use the empty template
    if not response_data:
        response_data = empty_template
    else:
        # Ensure we are always dealing with a full structure
        if "MonetizationData" not in response_data:
            response_data = {"MonetizationData": empty_template["MonetizationData"]}
    
    # Normalize the JSON response into a DataFrame
    result = pd.json_normalize(response_data)
    result['url'] = url  # Add the source URL to the DataFrame
    
    # Append the DataFrame to the list
    dfs.append(result)
    
    # Increment the counter
    count += 1
    
    # Break the loop after processing 5 URLs
    if count == 5:
        break


The extracted data in JSON format are as follows:

{
   "Donation Links": ["https://dons.vivementle9juin.fr"],
   "Donation Methods": [],
   "Suggested Donation Amounts": [],
   "Recurring Donations": "No",
   "Donation-Related Text": [],
   "Donation Buttons/Widgets": [],
   "Donation Section Location": "Menu bar",
   "Third-Party Platforms": [],
   "Call-to-Action": [],
   "Payment Platforms": [],
   "Payment Button/Widget Location": [],
   "Accepted Payment Methods": [],
   "Pricing Models": [],
   "Currency Support": [],
   "Subscription Plans": []
}

In the webpage, there is a donation link ("Faire un don") on the menu bar leading to "https://dons.vivementle9juin.fr". However, there are no clearly specified accepted donation methods, suggested donation amounts, recurrence options for donations, or specific call-to-action for donations. Likewise, there are no indications of third-party donation platforms, payment platforms, accepted payment methods, pricing models, currency support

In [187]:
if dfs:
    final_result = pd.concat(dfs, ignore_index=True)  # Combine all DataFrames into one
    print(final_result)  # Display the final combined DataFrame

  MonetizationData.DonationLinks MonetizationData.DonationMethods  \
0                             []                               []   
1                             []                               []   
2                             []                               []   
3                             []                               []   
4                             []                               []   

  MonetizationData.SuggestedDonationAmounts  \
0                                        []   
1                                        []   
2                                        []   
3                                        []   
4                                        []   

  MonetizationData.RecurringDonationsOption  \
0                                             
1                                             
2                                             
3                                             
4                                             

  MonetizationData.

In [188]:
final_result

,MonetizationData.DonationLinks,MonetizationData.DonationMethods,MonetizationData.SuggestedDonationAmounts,MonetizationData.RecurringDonationsOption,MonetizationData.TextRelatedToDonations,MonetizationData.ButtonsOrWidgets,MonetizationData.PlacementOfDonationSection,MonetizationData.ThirdPartyPlatforms,MonetizationData.CallToActionMessages,MonetizationData.AcceptedPaymentMethods,MonetizationData.PricingModels,MonetizationData.CurrencySupport,MonetizationData.SubscriptionPlans,url
0,[],[],[],,[],[],,[],[],[],[],[],[],https://vivementle9juin.fr/projet/europe-qui-r...
1,[],[],[],,[],[],,[],[],[],[],[],[],https://pravda-gr.com/world/2024/05/09/6152.html
2,[],[],[],,[],[],,[],[],[],[],[],[],https://triglavmedia.si/novice/okolje-ekologij...
3,[],[],[],,[],[],,[],[],[],[],[],[],https://euronewsbulgaria.com/news/27068/kiev-c...
4,[],[],[],,[],[],,[],[],[],[],[],[],https://www.dz-rs.si/wps/portal/Home/seje/evid...
